In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cpu
False


In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()

X = data.data
y = data.target

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(455, 30)
(114, 30)


In [3]:
from torch.utils.data import Dataset

class CancerDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [4]:
from torch.utils.data import DataLoader

train_dataset = CancerDataset(
    X_train,
    y_train
)

test_dataset = CancerDataset(
    X_test,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32
)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))

Train batches: 15
Test batches: 4


In [5]:
class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(30, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 2)

        )

    def forward(self, x):
        return self.network(x)

model = MLP()

print(model)

MLP(
  (network): Sequential(
    (0): Linear(in_features=30, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=32, out_features=2, bias=True)
  )
)


In [6]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = MLP().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print(device)

cpu


In [8]:
epochs = 20

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss = {running_loss:.4f}")

Epoch 1, Loss = 8.4858
Epoch 2, Loss = 5.5536
Epoch 3, Loss = 4.5590
Epoch 4, Loss = 3.7606
Epoch 5, Loss = 3.6859
Epoch 6, Loss = 2.9012
Epoch 7, Loss = 2.4825
Epoch 8, Loss = 2.1384
Epoch 9, Loss = 2.1595
Epoch 10, Loss = 2.1042
Epoch 11, Loss = 2.4317
Epoch 12, Loss = 1.7911
Epoch 13, Loss = 1.6785
Epoch 14, Loss = 1.7692
Epoch 15, Loss = 2.2666
Epoch 16, Loss = 1.7377
Epoch 17, Loss = 3.0505
Epoch 18, Loss = 1.8275
Epoch 19, Loss = 1.5007
Epoch 20, Loss = 1.6224


In [9]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)

        _, predicted = torch.max(outputs, 1)

        total += y_batch.size(0)

        correct += (predicted == y_batch).sum().item()

accuracy = 100 * correct / total

print("Accuracy =", accuracy)

Accuracy = 95.6140350877193
